# Navier-Stokes Flow Simulation


In [20]:
# Import required packages for finite element analysis
using Ferrite              # Main FE library
using SparseArrays         # Sparse matrix operations
using BlockArrays          # Block-structured matrices for velocity-pressure coupling
using LinearAlgebra        # Linear algebra operations
using UnPack               # Convenient unpacking of structs
using LinearSolve          # Linear solver interface
using WriteVTK             # VTK output for visualization
using OrdinaryDiffEq       # ODE/DAE time integration

# Physical parameters
ν = 1.0 / 1000.0  # Dynamic viscosity [Pa⋅s] - corresponds to water-like fluid


# Simulation parameters
dim = 2           # Spatial dimension


2

## 2. Mathematical Formulation and Weak Form Derivation

### 2.1 Strong Form of Incompressible Navier-Stokes Equations

The incompressible Navier-Stokes equations in their strong form are:

**Momentum equation:**
$$\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} - \nu \nabla^2 \mathbf{u} + \nabla p = \mathbf{f} \quad \text{in } \Omega \times (0,T]$$

**Continuity equation:**
$$\nabla \cdot \mathbf{u} = 0 \quad \text{in } \Omega \times (0,T]$$

where:
- $\mathbf{u}(\mathbf{x},t) = (u_x, u_y)^T$ is the velocity field
- $p(\mathbf{x},t)$ is the pressure field  
- $\nu$ is the kinematic viscosity
- $\mathbf{f}$ is the body force (assumed zero here)
- $\Omega \subset \mathbb{R}^2$ is the computational domain

### 2.2 Weak Formulation Derivation

To derive the weak form, we multiply the momentum equation by a test function $\mathbf{v} \in \mathcal{V}$ and the continuity equation by a test function $q \in \mathcal{Q}$, then integrate over the domain.

#### Step 1: Multiply by test functions and integrate

$$\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v} \, d\Omega + \int_\Omega [(\mathbf{u} \cdot \nabla)\mathbf{u}] \cdot \mathbf{v} \, d\Omega - \nu \int_\Omega \nabla^2 \mathbf{u} \cdot \mathbf{v} \, d\Omega + \int_\Omega \nabla p \cdot \mathbf{v} \, d\Omega = 0$$

$$\int_\Omega q \nabla \cdot \mathbf{u} \, d\Omega = 0$$

#### Step 2: Apply integration by parts to viscous and pressure terms

For the viscous term: $-\nu \int_\Omega \nabla^2 \mathbf{u} \cdot \mathbf{v} \, d\Omega = \nu \int_\Omega \nabla \mathbf{u} : \nabla \mathbf{v} \, d\Omega - \nu \int_{\partial\Omega} (\nabla \mathbf{u} \cdot \mathbf{n}) \cdot \mathbf{v} \, d\Gamma$

For the pressure term: $\int_\Omega \nabla p \cdot \mathbf{v} \, d\Omega = -\int_\Omega p \nabla \cdot \mathbf{v} \, d\Omega + \int_{\partial\Omega} p (\mathbf{v} \cdot \mathbf{n}) \, d\Gamma$

#### Step 3: Final weak form

Assuming homogeneous natural boundary conditions, the **weak form** becomes:

**Find** $(\mathbf{u}, p) \in \mathcal{V} \times \mathcal{Q}$ **such that** $\forall (\mathbf{v}, q) \in \mathcal{V}_0 \times \mathcal{Q}$:

$$\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v} \, d\Omega + \int_\Omega [(\mathbf{u} \cdot \nabla)\mathbf{u}] \cdot \mathbf{v} \, d\Omega + \nu \int_\Omega \nabla \mathbf{u} : \nabla \mathbf{v} \, d\Omega - \int_\Omega p \nabla \cdot \mathbf{v} \, d\Omega = 0$$

$$\int_\Omega q \nabla \cdot \mathbf{u} \, d\Omega = 0$$

### 2.3 Finite Element Discretization

We discretize the velocity using:

$$\mathbf{u}_h = \sum_{i=1}^{N_v} u_i \boldsymbol{\phi}_i(\mathbf{x}), \quad p_h = \sum_{j=1}^{N_p} p_j \psi_j(\mathbf{x})$$

This leads to the **semi-discrete system**:

$$\mathbf{M} \frac{d\mathbf{U}}{dt} + \mathbf{K}(\mathbf{U}) \mathbf{U} = \mathbf{0}$$

where:
- $\mathbf{U} = [\mathbf{u}^T, \mathbf{p}^T]^T$ is the solution vector
- $\mathbf{M}$ is the mass matrix (only velocity components)
- $\mathbf{K}(\mathbf{U})$ is the nonlinear system matrix

## 3. Mesh Generation and Geometry Setup

We create a 2D rectangular channel domain suitable for studying flow phenomena such as the development of the boundary layer and potential vortex shedding.

In [21]:
# Domain geometry parameters
L = 1.0    # Channel length [m]
H = 0.1    # Channel height [m]
println("Channel dimensions: L = ", L, " m, H = ", H, " m")
println("Aspect ratio L/H = ", L/H)

# Mesh parameters  
nels = (40, 20)  # Number of elements in (x, y) directions
left = Vec((0.0, 0.0))    # Lower-left corner
right = Vec((L, H))       # Upper-right corner

# Generate structured quadrilateral mesh
grid = generate_grid(Quadrilateral, nels, left, right)

# Create coordinate arrays for debugging/analysis
xgrid = Vector(range(0, L, step=L/nels[1]))
ygrid = Vector(range(0, H, step=H/nels[2]))

# Add vertex set for potential pressure constraint (removes pressure nullspace)
addvertexset!(grid, "corner", (x) -> x[1] ≈ 0.0 && x[2] ≈ 0.0)

println("Generated mesh with ", getncells(grid), " elements and ", getnnodes(grid), " nodes")
println("Element size: Δx ≈ ", L/nels[1], ", Δy ≈ ", H/nels[2])

Channel dimensions: L = 1.0 m, H = 0.1 m
Aspect ratio L/H = 10.0
Generated mesh with 800 elements and 861 nodes
Element size: Δx ≈ 0.025, Δy ≈ 0.005


## 4. Finite Element Spaces and DOF Handler Setup



In [22]:
# Define finite element interpolations
# Velocity: P3 Lagrange in 2D (vector-valued)
ip_v = Lagrange{RefQuadrilateral, 3}()^dim
println("Velocity interpolation: P3 Lagrange, ", getnbasefunctions(ip_v), " DOFs per element")

# Pressure: P2 Lagrange (scalar-valued)  
ip_p = Lagrange{RefQuadrilateral, 2}()
println("Pressure interpolation: P2 Lagrange, ", getnbasefunctions(ip_p), " DOFs per element")

# Geometry interpolation (linear for straight-sided elements)
ip_geo = Lagrange{RefQuadrilateral, 1}()

# Define quadrature rules
# Higher-order rule for accurate integration of nonlinear terms
qr = QuadratureRule{RefQuadrilateral}(5)  # 5×5 Gauss quadrature
qr_facet = FacetQuadratureRule{RefQuadrilateral}(2)  # For boundary integrals

# Create cell values (encapsulate shape functions, derivatives, and quadrature)
cellvalues_v = CellValues(qr, ip_v, ip_geo)
cellvalues_p = CellValues(qr, ip_p, ip_geo) 
facevalues_p = FacetValues(qr_facet, ip_p, ip_geo)

println("Quadrature points per cell: ", getnquadpoints(cellvalues_v))
println("Quadrature points per face: ", getnquadpoints(facevalues_p))

Velocity interpolation: P3 Lagrange, 32 DOFs per element
Pressure interpolation: P2 Lagrange, 9 DOFs per element
Quadrature points per cell: 25
Quadrature points per face: 2


In [23]:
# Setup DOF handler - manages global degree of freedom numbering
dh = DofHandler(grid)

# Add fields: velocity (vector) and pressure (scalar)
add!(dh, :v, ip_v)  # Velocity field with P3 interpolation
add!(dh, :p, ip_p)  # Pressure field with P2 interpolation

# Finalize DOF distribution
close!(dh)



DofHandler{2, Grid{2, Quadrilateral, Float64}}
  Fields:
    :v, Lagrange{RefQuadrilateral, 3}()^2
    :p, Lagrange{RefQuadrilateral, 2}()
  Dofs per cell: 41
  Total dofs: 18083

## 5. Boundary Conditions Implementation

We implement the following boundary conditions for channel flow:

### Mathematical Description:
- **Inlet (left)**: Parabolic velocity profile $\mathbf{u}(0,y,t) = (u_{\text{in}}(t) \cdot 4y(H-y)/H^2, 0)^T$
- **Outlet (right)**: Zero pressure $p(L,y,t) = 0$  
- **Walls (top/bottom)**: No-slip condition $\mathbf{u}(x,0,t) = \mathbf{u}(x,H,t) = \mathbf{0}$

The parabolic inlet profile ensures:
1. No-slip conditions at walls ($y=0,H$)
2. Maximum velocity at channel centerline ($y=H/2$)
3. Smooth velocity ramp-up for stability

In [24]:
# Initialize constraint handler for boundary conditions
ch = ConstraintHandler(dh)

# 1. No-slip boundary conditions on walls (top and bottom)
noslip_facets = ["bottom", "top"]
∂Ω_noslip = union(getfacetset.((grid,), noslip_facets)...)
noslip_bc = Dirichlet(:v, ∂Ω_noslip, (x, t) -> Vec((0.0, 0.0)), [1, 2])
add!(ch, noslip_bc)

# 2. Parabolic inlet velocity profile with time-dependent amplitude
vmax = 1.5  # Maximum inlet velocity [m/s]
vin(t) = min(t * vmax, vmax)  # Ramp up velocity over 1 second

# Parabolic profile: u(y) = u_max * 4y(H-y)/H^2
parabolic_inflow_profile(x, t) = Vec((vin(t) * 4 * x[2] * (H - x[2]) / H^2, 0.0))

inlet_facets = getfacetset(dh.grid, "left")
inlet_bc = Dirichlet(:v, inlet_facets, (x, t) -> parabolic_inflow_profile(x, t))
add!(ch, inlet_bc)

# 3. Zero pressure at outlet (removes pressure nullspace)
# outlet_facets = getfacetset(grid, "right")
# pressure_bc = Dirichlet(:p, outlet_facets, (x, t) -> 0.0)
# add!(ch, pressure_bc)
# mean_value_constraint = setup_mean_constraint(dh, facevalues_p)
# add!(ch, mean_value_constraint) 

# Finalize boundary conditions
close!(ch)

# Apply initial conditions
update!(ch, 0.0)

println("Boundary conditions applied:")
println("- No-slip walls: ", length(∂Ω_noslip), " facets")
println("- Parabolic inlet: ", length(inlet_facets), " facets") 
println("- Maximum inlet velocity: ", vmax, " m/s")

Boundary conditions applied:
- No-slip walls: 80 facets
- Parabolic inlet: 20 facets
- Maximum inlet velocity: 1.5 m/s


## 6. Mass Matrix Assembly

The mass matrix $\mathbf{M}$ arises from the time derivative term in the weak formulation:

$$\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v} \, d\Omega$$

### Discretized Form:
$$M_{ij} = \int_\Omega \boldsymbol{\phi}_i \cdot \boldsymbol{\phi}_j \, d\Omega$$

where $\boldsymbol{\phi}_i$ are the velocity shape functions.

### Matrix Structure:
The mass matrix has block structure:
$$\mathbf{M} = \begin{bmatrix} \mathbf{M}_{vv} & \mathbf{0} \\ \mathbf{0} & \mathbf{0} \end{bmatrix}$$

Only the velocity block $\mathbf{M}_{vv}$ is non-zero since pressure has no time derivative in the incompressible formulation.

In [25]:
function assemble_mass_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, 
                              M::SparseMatrixCSC, dh::DofHandler)
    """
    Assemble the mass matrix M for the velocity time derivative term.
    
    Mathematical form: M_ij = ∫_Ω φᵢ ⋅ φⱼ dΩ (only for velocity DOFs)
    """
    
    # Get dimensions for block structure
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p) 
    n_basefuncs = n_basefuncs_v + n_basefuncs_p
    
    # Block indices: 1 = velocity, 2 = pressure
    v▄, p▄ = 1, 2
    
    # Local element matrix with block structure
    Mₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs), 
                     [n_basefuncs_v, n_basefuncs_p], 
                     [n_basefuncs_v, n_basefuncs_p])
    
    # Initialize assembler
    mass_assembler = start_assemble(M)
    
    # Element loop
    for cell in CellIterator(dh)
        fill!(Mₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        
        # Quadrature loop
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)  # Jacobian determinant × weight
            
            # Velocity mass matrix: ∫ φᵢ ⋅ φⱼ dΩ
            for i in 1:n_basefuncs_v
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    # Vector dot product for velocity components
                    Mₑ[BlockIndex((v▄, v▄), (i, j))] += φᵢ ⋅ φⱼ * dΩ
                end
            end
            # Note: Pressure block remains zero (no ∂p/∂t term)
        end
        
        # Assemble element matrix into global matrix
        assemble!(mass_assembler, celldofs(cell), Mₑ)
    end
    
    return M
end

# Allocate and assemble mass matrix
M = allocate_matrix(dh)
M = assemble_mass_matrix(cellvalues_v, cellvalues_p, M, dh)

println("Mass matrix assembled:")
println("- Size: ", size(M))
println("- Non-zeros: ", nnz(M))
println("- Sparsity: ", nnz(M)/(size(M,1)*size(M,2))*100, "%")

Mass matrix assembled:
- Size: (18083, 18083)
- Non-zeros: 1165129
- Sparsity: 0.3563141318701162%


## 7. Stokes Matrix Assembly

The Stokes matrix $\mathbf{K}$ contains the linear terms from the weak formulation:

### Viscous Term:
$$\nu \int_\Omega \nabla \mathbf{u} : \nabla \mathbf{v} \, d\Omega \quad \Rightarrow \quad K^{visc}_{ij} = \nu \int_\Omega \nabla \boldsymbol{\phi}_i : \nabla \boldsymbol{\phi}_j \, d\Omega$$

### Pressure-Velocity Coupling:
$$-\int_\Omega p \nabla \cdot \mathbf{v} \, d\Omega \,and \int_\Omega q \nabla \cdot \mathbf{u} \, d\Omega$$

$$\Rightarrow \quad K^{pres}_{ij} = \int_\Omega \psi_i \nabla \cdot \boldsymbol{\phi}_j \, d\Omega$$

### Saddle-Point Structure:
$$\mathbf{K} = \begin{bmatrix} 
-\nu \mathbf{A} & \mathbf{B}^T \\
\mathbf{B} & \mathbf{0}
\end{bmatrix}$$

where:
- $\mathbf{A}$: velocity stiffness matrix (negative for sign convention)
- $\mathbf{B}$: divergence matrix  
- $\mathbf{B}^T$: gradient matrix

In [26]:
function assemble_stokes_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, 
                              ν, K::SparseMatrixCSC, dh::DofHandler)
    """
    Assemble the Stokes matrix containing viscous and pressure coupling terms.
    
    Mathematical form:
    K = [[-ν∇²    ∇  ]
         [ ∇⋅     0  ]]
    """
    
    # Get dimensions for block structure
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs = n_basefuncs_v + n_basefuncs_p
    
    # Block indices
    v▄, p▄ = 1, 2
    
    # Local element matrix
    Kₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs),
                     [n_basefuncs_v, n_basefuncs_p],
                     [n_basefuncs_v, n_basefuncs_p])
    
    # Initialize assembler
    stiffness_assembler = start_assemble(K)
    
    # Element loop
    for cell in CellIterator(dh)
        fill!(Kₑ, 0)
        
        # Update shape functions for current element
        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_p, cell)
        
        # Quadrature loop
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            
            # Viscous term: -ν ∫ ∇uᵢ : ∇vⱼ dΩ
            for i in 1:n_basefuncs_v
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    ∇φⱼ = shape_gradient(cellvalues_v, q_point, j)
                    # Double contraction: ∇φᵢ : ∇φⱼ = ∑ₖₗ ∂φᵢᵏ/∂xₗ ∂φⱼᵏ/∂xₗ
                    Kₑ[BlockIndex((v▄, v▄), (i, j))] -= ν * ∇φᵢ ⊡ ∇φⱼ * dΩ
                end
            end
            
            # Pressure-velocity coupling: ∫ p ∇⋅v dΩ and ∫ q ∇⋅u dΩ
            for j in 1:n_basefuncs_p
                ψ = shape_value(cellvalues_p, q_point, j)  # Pressure shape function
                for i in 1:n_basefuncs_v
                    divφ = shape_divergence(cellvalues_v, q_point, i)  # ∇⋅φᵢ
                    
                    # Gradient term: ∫ ψ ∇⋅φᵢ dΩ
                    Kₑ[BlockIndex((v▄, p▄), (i, j))] += (divφ * ψ) * dΩ
                    
                    # Divergence term: ∫ φᵢ ∇⋅ψ dΩ = ∫ ψ ∇⋅φᵢ dΩ (symmetry)
                    Kₑ[BlockIndex((p▄, v▄), (j, i))] += (ψ * divφ) * dΩ
                end
            end
        end
        
        # Assemble element matrix into global matrix
        assemble!(stiffness_assembler, celldofs(cell), Kₑ)
    end
    
    return K
end

# Allocate and assemble Stokes matrix
K = allocate_matrix(dh)
K = assemble_stokes_matrix(cellvalues_v, cellvalues_p, ν, K, dh)

println("Stokes matrix assembled:")
println("- Size: ", size(K))
println("- Non-zeros: ", nnz(K))
println("- Condition estimate: saddle-point (indefinite)")
println("- Viscosity parameter: ν = ", ν)

Stokes matrix assembled:
- Size: (18083, 18083)
- Non-zeros: 1165129
- Condition estimate: saddle-point (indefinite)
- Viscosity parameter: ν = 0.001


## 8. Nonlinear Term Implementation

The nonlinear convection term represents the material derivative:

$$\int_\Omega [(\mathbf{u} \cdot \nabla)\mathbf{u}] \cdot \mathbf{v} \, d\Omega$$

### Discretized Form:
$$N(\mathbf{u})_i = \int_\Omega [(\mathbf{u}_h \cdot \nabla)\mathbf{u}_h] \cdot \boldsymbol{\phi}_i \, d\Omega$$

where $\mathbf{u}_h = \sum_j u_j \boldsymbol{\phi}_j$ is the discrete velocity field.

### Expanded Form:
$$(\mathbf{u} \cdot \nabla)\mathbf{u} = \begin{bmatrix} 
u \frac{\partial u}{\partial x} + v \frac{\partial u}{\partial y} \\
u \frac{\partial v}{\partial x} + v \frac{\partial v}{\partial y}
\end{bmatrix}$$

This term is **nonlinear** and requires iterative solution (Newton's method).

In [27]:
function navierstokes_rhs_element!(dvₑ, vₑ, cellvalues_v)
    """
    Compute element contribution to nonlinear convection term: -∫(u⋅∇u)⋅v dΩ
    
    Input:
    - vₑ: element velocity DOFs  
    - cellvalues_v: velocity shape functions and gradients
    
    Output: 
    - dvₑ: element residual contribution (modified in-place)
    """
    
    n_basefuncs = getnbasefunctions(cellvalues_v)
    
    # Quadrature loop
    for q_point in 1:getnquadpoints(cellvalues_v)
        dΩ = getdetJdV(cellvalues_v, q_point)
        
        # Evaluate velocity and velocity gradient at quadrature point
        ∇v = function_gradient(cellvalues_v, q_point, vₑ)  # ∇uₕ
        v = function_value(cellvalues_v, q_point, vₑ)      # uₕ
        
        # Compute convection: (u⋅∇)u = u⊗∇u where ⊗ is tensor product
        # Note: ∇v is a tensor, v⋅∇v' gives the convection vector
        convection = v ⋅ ∇v'  # Vector: [(u⋅∇)u, (u⋅∇)v]ᵀ
        
        # Integrate against test functions
        for j in 1:n_basefuncs
            φⱼ = shape_value(cellvalues_v, q_point, j)
            
            # Residual: -∫ [(u⋅∇)u] ⋅ φⱼ dΩ (negative for RHS formulation)
            dvₑ[j] -= convection ⋅ φⱼ * dΩ
        end
    end
    return
end

println("Nonlinear convection term implemented")
println("- Form: -∫ (u⋅∇u)⋅v dΩ")  
println("- Requires: velocity field evaluation at quadrature points")
println("- Returns: element residual vector")

Nonlinear convection term implemented
- Form: -∫ (u⋅∇u)⋅v dΩ
- Requires: velocity field evaluation at quadrature points
- Returns: element residual vector


In [28]:
# Define parameters structure for ODE system
struct RHSparams
    K::SparseMatrixCSC         # Stokes matrix
    ch::ConstraintHandler      # Boundary conditions  
    dh::DofHandler            # DOF management
    cellvalues_v::CellValues  # Velocity shape functions
    u::Vector                 # Workspace for solution vector
end

# Initialize parameters
u₀ = zeros(ndofs(dh))       # Initial solution
apply!(u₀, ch)              # Apply initial boundary conditions
p = RHSparams(K, ch, dh, cellvalues_v, copy(u₀))

function navierstokes!(du, u_uc, p::RHSparams, t)
    """
    Compute the full Navier-Stokes residual: du/dt = -K*u - N(u)
    
    This function combines:
    1. Linear Stokes terms: -K*u
    2. Nonlinear convection: -N(u) 
    3. Boundary condition enforcement
    """
    
    @unpack K, ch, dh, cellvalues_v, u = p
    
    # Update solution and boundary conditions
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    
    # Linear contribution: du = K * u (Stokes operator)
    mul!(du, K, u) 
    
    # Add nonlinear convection term
    v_range = dof_range(dh, :v)  # Velocity DOF indices
    n_basefuncs = getnbasefunctions(cellvalues_v)
    vₑ = zeros(n_basefuncs)      # Element velocity vector
    duₑ = zeros(n_basefuncs)     # Element residual vector
    
    # Loop over all elements
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell)
        
        # Extract velocity DOFs for this element
        v_celldofs = @view celldofs(cell)[v_range]
        vₑ .= @views u[v_celldofs]
        
        # Compute nonlinear contribution
        fill!(duₑ, 0.0)
        navierstokes_rhs_element!(duₑ, vₑ, cellvalues_v)
        
        # Assemble into global residual
        assemble!(du, v_celldofs, duₑ)
    end
    
    return
end

println("Global Navier-Stokes residual function implemented")
println("- Combines linear Stokes operator with nonlinear convection")
println("- Enforces boundary conditions at each time step")
println("- Returns: time derivative du/dt")

Global Navier-Stokes residual function implemented
- Combines linear Stokes operator with nonlinear convection
- Enforces boundary conditions at each time step
- Returns: time derivative du/dt


## 9. Jacobian Matrix Assembly

For implicit time integration, we need the Jacobian matrix of the nonlinear residual:

$$\mathbf{J} = \frac{\partial \mathbf{F}}{\partial \mathbf{u}} = \mathbf{K} + \frac{\partial \mathbf{N}(\mathbf{u})}{\partial \mathbf{u}}$$

### Derivative of Convection Term:
$$\frac{\partial}{\partial \mathbf{u}} [(\mathbf{u} \cdot \nabla)\mathbf{u}] = (\delta\mathbf{u} \cdot \nabla)\mathbf{u} + (\mathbf{u} \cdot \nabla)\delta\mathbf{u}$$

### Element Jacobian:
$$J_{ij} = \int_\Omega [(\boldsymbol{\phi}_i \cdot \nabla)\mathbf{u}_h + (\mathbf{u}_h \cdot \nabla)\boldsymbol{\phi}_i] \cdot \boldsymbol{\phi}_j \, d\Omega$$


In [29]:
function navierstokes_jac_element!(Jₑ, vₑ, cellvalues_v)
    """
    Compute element Jacobian of nonlinear convection term.
    
    Mathematical form:
    J_ij = ∫ [(φᵢ⋅∇)u + (u⋅∇)φᵢ] ⋅ φⱼ dΩ
    
    This represents the linearization: d/du[(u⋅∇)u]⋅δu
    """
    
    n_basefuncs = getnbasefunctions(cellvalues_v)
    
    # Quadrature loop
    for q_point in 1:getnquadpoints(cellvalues_v)
        dΩ = getdetJdV(cellvalues_v, q_point)
        
        # Evaluate current velocity and gradient
        ∇v = function_gradient(cellvalues_v, q_point, vₑ)  # ∇uₕ  
        v = function_value(cellvalues_v, q_point, vₑ)      # uₕ
        
        # Test function loop
        for j in 1:n_basefuncs
            φⱼ = shape_value(cellvalues_v, q_point, j)
            
            # Trial function loop  
            for i in 1:n_basefuncs
                φᵢ = shape_value(cellvalues_v, q_point, i)
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                
                # Linearized convection terms:
                # 1. (φᵢ⋅∇)u: trial function advecting current velocity  
                # 2. (u⋅∇)φᵢ: current velocity advecting trial function
                linearized_conv = (φᵢ ⋅ ∇v' + v ⋅ ∇φᵢ')
                
                # Integrate: -∫ linearized_conv ⋅ φⱼ dΩ  
                Jₑ[j, i] -= linearized_conv ⋅ φⱼ * dΩ
            end
        end
    end
    return
end

function navierstokes_jac!(J, u_uc, p, t)
    """
    Assemble full Jacobian matrix: J = K + ∂N/∂u
    """
    
    @unpack K, ch, dh, cellvalues_v, u = p
    
    # Update solution and boundary conditions
    u .= u_uc  
    update!(ch, t)
    apply!(u, ch)
    
    # Initialize with Stokes matrix: J = K
    nonzeros(J) .= nonzeros(K)
    
    # Add nonlinear Jacobian contribution
    assembler = start_assemble(J; fillzero = false)
    
    n_basefuncs = getnbasefunctions(cellvalues_v)
    Jₑ = zeros(n_basefuncs, n_basefuncs)  # Element Jacobian
    vₑ = zeros(n_basefuncs)               # Element velocity
    v_range = dof_range(dh, :v)
    
    # Element loop
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell)
        v_celldofs = @view celldofs(cell)[v_range]
        
        # Extract element velocity
        vₑ .= @views u[v_celldofs]
        
        # Compute nonlinear Jacobian contribution
        fill!(Jₑ, 0.0)
        navierstokes_jac_element!(Jₑ, vₑ, cellvalues_v)
        
        # Assemble into global matrix
        assemble!(assembler, v_celldofs, Jₑ)
    end
    
    # Apply boundary conditions to Jacobian
    return apply!(J, ch)
end

# Create sparsity pattern for Jacobian (same as Stokes matrix)
jac_sparsity = sparse(K)

println("Jacobian assembly implemented:")
println("- Linear part: Stokes matrix K")
println("- Nonlinear part: linearized convection ∂N/∂u") 
println("- Total entries: ", nnz(jac_sparsity))
println("- Required for implicit time integration")

Jacobian assembly implemented:
- Linear part: Stokes matrix K
- Nonlinear part: linearized convection ∂N/∂u
- Total entries: 1165129
- Required for implicit time integration


## 10. Time Integration Setup

We solve the **differential-algebraic equation (DAE)** system:

$$\mathbf{M} \frac{d\mathbf{U}}{dt} = -\mathbf{F}(\mathbf{U}, t)$$

where $\mathbf{F}(\mathbf{U}, t) = \mathbf{K}\mathbf{U} + \mathbf{N}(\mathbf{U})$ is the spatial residual.


In [30]:
# Simulation time parameters
T = 6.0        # Final simulation time [s]
Δt₀ = 0.001    # Initial time step [s]  
Δt_save = 0.1  # Output frequency [s]

# Apply boundary conditions to mass matrix (for DAE structure)
apply!(M, ch)

# Define step limiter for boundary condition enforcement
function ferrite_limiter!(u, _, p, t)
    """Enforce boundary conditions during time integration."""
    update!(p.ch, t)
    return apply!(u, p.ch)
end

# Create ODE function with mass matrix (DAE formulation)
rhs = ODEFunction(navierstokes!; 
                 mass_matrix = M,           # Mass matrix for DAE  
                 jac = navierstokes_jac!,   # Jacobian for implicit method
                 jac_prototype = jac_sparsity)  # Sparsity pattern

# Define ODE problem: M*du/dt = -F(u,t)  
problem = ODEProblem(rhs, u₀, (0.0, T), p)

# Custom norm that only considers free DOFs (unconstrained variables)
struct FreeDofErrorNorm
    ch::ConstraintHandler
end

# Scalar norm (fallback)
(fe_norm::FreeDofErrorNorm)(u::Union{AbstractFloat, Complex}, t) = 
    DiffEqBase.ODE_DEFAULT_NORM(u, t)

# Vector norm (only free DOFs)
(fe_norm::FreeDofErrorNorm)(u::AbstractArray, t) = 
    DiffEqBase.ODE_DEFAULT_NORM(u[fe_norm.ch.free_dofs], t)

println("Time integration setup completed:")
println("- Final time: T = ", T, " s")
println("- Initial time step: Δt₀ = ", Δt₀, " s") 
println("- DAE structure: differential (velocity) + algebraic (pressure)")
println("- Mass matrix size: ", size(M))
println("- Free DOFs for error norm: ", length(ch.free_dofs))

Time integration setup completed:
- Final time: T = 6.0 s
- Initial time step: Δt₀ = 0.001 s
- DAE structure: differential (velocity) + algebraic (pressure)
- Mass matrix size: (18083, 18083)
- Free DOFs for error norm: 17481


## 11. Solver Configuration and Execution

We initialize the **Rodas5P** time integrator with adaptive time stepping and execute the simulation. The solver automatically adjusts time step size based on local error estimates while maintaining stability.

In [31]:
# Configure Rodas5P time integrator
timestepper = Rodas5P(
    autodiff = false,                    # Use provided Jacobian (not automatic differentiation)
    step_limiter! = ferrite_limiter!     # Enforce boundary conditions
)

# Initialize integrator with adaptive time stepping
integrator = init(
    problem, timestepper;
    initializealg = NoInit(),            # Don't modify initial conditions
    dt = Δt₀,                           # Initial time step  
    adaptive = true,                     # Enable adaptive time stepping
    abstol = 1.0e-4,                    # Absolute tolerance
    reltol = 1.0e-5,                    # Relative tolerance  
    progress = true,                     # Show progress bar
    progress_steps = 1,                  # Progress update frequency
    verbose = true,                      # Detailed output
    internalnorm = FreeDofErrorNorm(ch), # Custom error norm
    d_discontinuities = [1.0]            # Velocity ramp ends at t=1s
)

println("Integrator initialized with:")
println("- Method: Rodas5P (5th-order Rosenbrock)")  
println("- Adaptive time stepping: enabled")
println("- Absolute tolerance: ", 1.0e-4)
println("- Relative tolerance: ", 1.0e-5) 
println("- Velocity ramp discontinuity at t = 1.0 s")
println("\nStarting time integration...")

Integrator initialized with:
- Method: Rodas5P (5th-order Rosenbrock)
- Adaptive time stepping: enabled
- Absolute tolerance: 0.0001
- Relative tolerance: 1.0e-5
- Velocity ramp discontinuity at t = 1.0 s

Starting time integration...


## 12. Results Visualization and Export

The final step exports the simulation results in VTK format for visualization in ParaView. The solution at each time step contains both velocity and pressure fields defined on the finite element mesh.

In [32]:
# Create ParaView collection for time series visualization
pvd = paraview_collection("channel_flow")

# Time integration loop with output generation
println("Executing time integration and generating output...")
for (step, (u, t)) in enumerate(intervals(integrator))
    
    # Create VTK file for current time step
    VTKGridFile("channel_flow-$step", dh) do vtk
        # Write solution fields (velocity and pressure) to VTK
        write_solution(vtk, dh, u)
        
        # Add to time series collection
        pvd[t] = vtk
    end
    
    # Progress information
    if step % 10 == 0
        println("Completed time step $step, t = $(round(t, digits=3)) s")
    end
end

# Save the ParaView collection file
vtk_save(pvd)

println("\nSimulation completed successfully!")
println("Results exported to VTK format:")
println("- Time series file: channel_flow.pvd") 
println("- Individual time steps: channel_flow-*.vtu")
println("- Open channel_flow.pvd in ParaView for visualization")

# Display final simulation statistics
final_time = integrator.t
total_steps = length(integrator.sol.t)


Executing time integration and generating output...
Completed time step 10, t = 0.031 s
Completed time step 20, t = 0.525 sCompleted time step 20, t = 0.525 s
Completed time step 30, t = 1.037 s
Completed time step 40, t = 1.527 s

Simulation completed successfully!
Completed time step 30, t = 1.037 s
Completed time step 40, t = 1.527 s

Simulation completed successfully!
Results exported to VTK format:
- Time series file: channel_flow.pvd
- Individual time steps: channel_flow-*.vtu
- Open channel_flow.pvd in ParaView for visualization

Results exported to VTK format:
- Time series file: channel_flow.pvd
- Individual time steps: channel_flow-*.vtu
- Open channel_flow.pvd in ParaView for visualization


47